# 🎓 PALABRIA – Ejecución en Google Colab

Este notebook permite ejecutar la aplicación **PALABRIA** directamente en Google Colab, exponiéndola a través de una URL pública con **ngrok**.

### Requisitos previos
- Una cuenta gratuita en [ngrok.com](https://dashboard.ngrok.com/signup) para obtener tu **Auth Token**.
- Activa el entorno de ejecución con **GPU** (`Entorno de ejecución → Cambiar tipo de entorno de ejecución → T4 GPU`).

### Privacidad
- Cada usuario crea su propia base de datos en su Google Drive.
- Los datos **NO** se comparten con los autores del proyecto.

---
## 1️⃣ Configurar token de ngrok
Introduce tu token de autenticación de ngrok. Puedes obtenerlo en: https://dashboard.ngrok.com/get-started/your-authtoken

In [3]:
from getpass import getpass

#NGROK_TOKEN = getpass("🔑 Pega tu NGROK AUTHTOKEN: ").strip()
NGROK_TOKEN ="39G81fo3cFIz5wfI5EdevdgMNip_3Efmg5iNfMEcZuh3TvETL"
if not NGROK_TOKEN:
    raise ValueError("⚠️ Debes introducir un token de ngrok válido.")

!pip install -q pyngrok
from pyngrok import ngrok
ngrok.set_auth_token(NGROK_TOKEN)
ngrok.kill()  # limpiar túneles previos

print("✅ Token de ngrok configurado correctamente.")

✅ Token de ngrok configurado correctamente.


---
## 2️⃣ Obtener el proyecto
Elige **una** de las dos opciones:

### Cargar desde Google Drive
Usa esta celda si ya tienes el proyecto en tu Drive.

In [4]:
from google.colab import drive
import os, sys

drive.mount('/content/drive')

PROJECT = "/content/drive/MyDrive/Asistente-Educacion"  # ← Ajusta esta ruta
#
if not os.path.exists(PROJECT):
    raise FileNotFoundError(f"No se encontró la carpeta en: {PROJECT}")
#
os.chdir(PROJECT)
sys.path.insert(0, PROJECT)

print(f"✅ Proyecto cargado desde Drive: {PROJECT}")

Mounted at /content/drive
✅ Proyecto cargado desde Drive: /content/drive/MyDrive/Asistente-Educacion


---
## 3️⃣ Instalar dependencias

In [5]:
!pip install -q -r requirements.txt
!pip install -q nest_asyncio

print("\n✅ Dependencias instaladas.")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 4.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.1/68.1 kB 8.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 6.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 27.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 132.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 10.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 138.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.5/21.5 MB 81.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 27.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 78.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.1/17.1 MB 78.7 MB/s eta 0:00:00
   ━━━━━━━━━━

In [6]:
# Descargar modelo lingüístico de spaCy para español
!python -m spacy download es_core_news_lg

print("\n✅ Modelo de spaCy descargado.")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 568.0/568.0 MB 3.1 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('es_core_news_lg')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.

✅ Modelo de spaCy descargado.


---
## 4️⃣ Configurar base de datos

In [7]:
import os, pathlib

# La BD se guarda dentro del proyecto (también funciona en Drive)
DB_PATH = os.path.join(os.getcwd(), "data", "palabria.db")
pathlib.Path(os.path.dirname(DB_PATH)).mkdir(parents=True, exist_ok=True)

os.environ["DB_PATH"] = DB_PATH

print(f"✅ Base de datos configurada en: {DB_PATH}")

✅ Base de datos configurada en: /content/drive/MyDrive/Asistente-Educacion/data/palabria.db


---
## 5️⃣ Lanzar la aplicación
Esta celda levanta el backend FastAPI con uvicorn y crea un túnel ngrok público.

> ⏳ **El primer arranque tarda varios minutos** porque descarga el modelo LLM (~4 GB).

In [8]:
import os, sys, time, subprocess, socket, requests
import nest_asyncio
from pyngrok import ngrok

nest_asyncio.apply()

PORT = 8000
PROJECT = os.getcwd()

# ── Funciones auxiliares ───────────────────────────────────────────
def is_port_open(host, port, timeout=1):
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
        s.settimeout(timeout)
        try:
            s.connect((host, port))
            return True
        except (socket.error, socket.timeout):
            return False

def cleanup():
    """Cerrar túneles y procesos anteriores."""
    for t in ngrok.get_tunnels():
        ngrok.disconnect(t.public_url)
    subprocess.run(["pkill", "-f", "uvicorn"],
                   stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

# ── Limpiar ejecuciones previas ────────────────────────────────────
cleanup()
time.sleep(2)

# ── Levantar FastAPI en segundo plano ─────────────────────────────
proc = subprocess.Popen(
    [sys.executable, "-m", "uvicorn", "backend.main:app",
     "--host", "0.0.0.0", "--port", str(PORT)],
    cwd=PROJECT
)

# ── Esperar a que el backend esté disponible ──────────────────────
TIMEOUT = 300  # 5 min (la descarga del modelo puede tardar)
start = time.time()
print("⏳ Esperando a que el backend se inicie...")

while not is_port_open("127.0.0.1", PORT):
    if time.time() - start > TIMEOUT:
        proc.terminate()
        raise RuntimeError("❌ El backend no arrancó a tiempo.")
    time.sleep(2)

status = requests.get(f"http://127.0.0.1:{PORT}/status/").json()
print(f"✅ Backend activo  (modelo_listo: {status.get('modelo_listo', '?')})")

# ── Crear túnel ngrok ─────────────────────────────────────────────
tunnel = ngrok.connect(PORT, bind_tls=True)
PUBLIC_URL = tunnel.public_url

os.environ["BACKEND_URL"] = PUBLIC_URL

print()
print("═" * 55)
print("  🎓 PALABRIA está en marcha")
print("═" * 55)
print(f"  🌐 App :  {PUBLIC_URL}")
print(f"  📚 Docs:  {PUBLIC_URL}/docs")
print("═" * 55)
print()
print("📌 Mantén esta celda ejecutándose.")
print("   Para detener la app, interrumpe el kernel.")

⏳ Esperando a que el backend se inicie...
✅ Backend activo  (modelo_listo: False)

═══════════════════════════════════════════════════════
  🎓 PALABRIA está en marcha
═══════════════════════════════════════════════════════
  🌐 App :  https://lumberingly-subtriquetrous-iesha.ngrok-free.dev
  📚 Docs:  https://lumberingly-subtriquetrous-iesha.ngrok-free.dev/docs
═══════════════════════════════════════════════════════

📌 Mantén esta celda ejecutándose.
   Para detener la app, interrumpe el kernel.


---
## 🛠️ Utilidades (opcional)

### Detener todos los procesos
Ejecuta esta celda si necesitas parar la app sin reiniciar el entorno.

In [ ]:
import subprocess
from pyngrok import ngrok

# Cerrar túneles ngrok
for t in ngrok.get_tunnels():
    ngrok.disconnect(t.public_url)
ngrok.kill()

# Matar procesos de uvicorn
subprocess.run(["pkill", "-f", "uvicorn"],
               stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

print("✅ Todos los procesos detenidos.")

✅ Todos los procesos detenidos.


In [ ]:
!pkill -f ngrok

### Comprobar estado del backend

In [ ]:
import requests

try:
    r = requests.get("http://127.0.0.1:8000/status/", timeout=5)
    print("Estado del backend:", r.json())
except Exception as e:
    print(f"⚠️ Backend no disponible: {e}")

Estado del backend: {'modelo_listo': False, 'progress': 0, 'message': 'Modelo no cargado.'}
